In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

# <span style="color:red">ch1_허깅페이스</span>
- Transformers 라이브러리 내 pipeline() 함수
- Inference API(회원가입과 Access키가 있어야 함)
## 1. 텍스트 기반 감정분석(긍정/부정)

In [ ]:
from transformers import pipeline
classifier = pipeline(task="text-classification",
                      model="distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a HuggingFace course my whole life.")

In [ ]:
classifier("이 영화는 정말 최악이야. 쓰레기같은 영화야. 너도 꼭 봤으면 좋겠어. 이 재미없는 영화를.")

In [ ]:
result = classifier(["I've been waiting for a HuggingFace course my whole life.",
                    "I hate this so much!"])
[r.get('label') for r in result]

In [ ]:
classifier = pipeline(task="text-classification",
                      model="distilbert-base-uncased-finetuned-sst-2-english")
classifier(["I've been waiting for a HuggingFace course my whole life.",
            "I hate this so much!"])

# 2. 제로-샷 분류(zero-shot-classification)
- 비지도 학습

In [ ]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                     "facebook/bart-large-mnli")
classifier("I have a problem with my iphone that needs to be resolved asap!!",
          candidate_labels=["phone", "urgent", "tablet", "computer"])

In [ ]:
# 제시된 문장이 어떤 문장인지
classifier(
    "This is a course about the transformers library.",
    candidate_labels=["education", "business", "politics"]
)

# 3. text 생성

In [ ]:
generator = pipeline(task="text-generation",
                    model="gpt2") # 허깅페이스에는 gpt2까지
generator("In this course. We will teach you how to",
         pad_token_id=generator.tokenizer.eos_token_id)

In [ ]:
generator("이 과정은 다음과 같은 방법을 알려드려요. ",
         pad_token_id=generator.tokenizer.eos_token_id)

# 4. 마스크 채우기

In [ ]:
unmasker = pipeline("fill-mask", "distilroberta-base")
unmasker("I'm going to hospital and meet a <mask>")

In [ ]:
unmasker("Hello, I'm a <mask> girl",
        top_k=2) #top_k를 안 주면 5개

In [ ]:
# google-bert/bert-base-uncased사용을 위해 key 발부
from transformers import pipeline
unmasker = pipeline(task="fill-mask",
                   model="google-bert/bert-base-uncased")
unmasker("Hello, I'm a [MASK] model", top_k=2)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
# print(os.environ['HF_TOKEN'])

In [ ]:
from huggingface_hub import InferenceClient
client = InferenceClient(provider="hf-inference",
                         api_key=os.environ['HF_TOKEN'])
result = client.fill_mask(
    "Hello, I'm a [MASK] model",
    model="google-bert/bert-base-uncased",
    top_k=2
)

In [ ]:
[r.sequence for r in result]

In [ ]:
# 다국어지원 모델도 한글 지원 만족스럽지 않을 수 있음
unmasker = pipeline("fill-mask",
                   model="bert-base-multilingual-cased") 

In [ ]:
unmasker("안녕하세요? 나는 [MASK] 모델입니다", top_k=3)

# 5. 개체명 인식(NER : Named Entity REcognition)

In [ ]:
ner = pipeline(task="ner")

In [ ]:
ner("My name is Sylvain and I work at Hugging Face in Brookly")

# 6. 질의 응답

In [ ]:
from transformers import pipeline
question_answer = pipeline("question-answering",
                          "distilbert-base-cased-distilled-squad")

In [ ]:
question_answer(
    question="Where do I work?",
    context="My name is Tom and I work at Facebook in Brooklyn"
    
)

In [ ]:
context="My name is Tom and I work at Facebook in Brooklyn"
context[29:37]

# 7. 문서요약
- 현재 torch 2.6이상 추천

In [ ]:
import torch
torch.__version__

In [ ]:
summarizer = pipeline(task="summarization",
                     model="facebook/bart-large-cnn")

In [ ]:
summarizer("""It is a momentous occasion for fans of the K-pop group BTS. The seven singers of the popular K-pop band plan to reunite as a group sometime in 2025 now that they’ve finished their service.
Last week, BTS superstars RM and V were discharged from South Korea’s military after fulfilling their mandatory service. Jimin and Jung Kook were discharged a day later. All four were enlisted in December 2023.
K-pop supergroup BTS could soon make a comeback with six out of its seven members discharged from South Korea’s military
Six of the group’s seven members served in the army, while Suga fulfilled his duty as a social service agent, an alternative form of military service.
Jin, the oldest BTS member, was discharged in June 2024. J-Hope was discharged in October.
In South Korea, all able-bodied men aged 18 to 28 are required by law to perform 18-21 months of military service under a conscription system meant to deter aggression from rival North Korea.
The law gives special exemptions to athletes, classical and traditional musicians, and ballet and other dancers if they have obtained top prizes in certain competitions and are assessed to have enhanced national prestige. K-pop stars and other entertainers aren’t subject to such privileges.
However, in 2020, BTS postponed their service until age 30 after South Korea’s National Assembly revised its Military Service Act, allowing K-pop stars to delay their enlistment until age 30.
There was heated public debate in 2022 over whether to offer special exemptions of mandatory military service for BTS members, 
until the group’s management agency announced in October 2022 that all seven members would fulfill their 
duties.""",
           max_length=150,
           min_length=30,
           do_sample=False # True면 무작위 단어로 창의적으로 요약
)

# 8. 번역

In [ ]:
# 한 -> 영
ko2en = pipeline("translation",
                 model="Helsinki-NLP/opus-mt-ko-en")

# 영 -> 한(?)
en2ko = pipeline("translation",
                 model="Helsinki-NLP/opus-mt-tc-big-en-ko")

In [ ]:
# 테스트 문장
ko_sentence = "이 문장을 영어로 번역해 주세요."
en_sentence = "I enjoy learning about AI."
ko_result = ko2en(ko_sentence)[0]['translation_text']
en_result = en2ko(en_sentence)[0]['translation_text']
print("한->영 :", ko_result)
print("영->한 :", en_result)

In [ ]:
result = ko2en([
    "이 문장을 영어로 번역해 주세요.",
    "내일은 드디어 LLM 시작!",
    "머신러닝과 딥러닝 평가가 있어요"
])

In [ ]:
result

In [ ]:
print('\n'.join([r['translation_text'] for r in result]))

# 9. 이미지를 설명하는 텍스트 생성

In [ ]:
imagetotext = pipeline(task="image-to-text",
                      model="ydshieh/vit-gpt2-coco-en")

In [ ]:
url = 'https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png'
from PIL import Image
import requests
image = Image.open(requests.get(url, stream=True).raw)
small_image = image.resize((150,150))
small_image

In [ ]:
imagetotext(url, max_new_tokens=30)

In [ ]:
type(small_image), type(image)

In [ ]:
imagetotext(image, max_new_tokens=50)

In [ ]:
# 내 PC의 이미지 설명
image = Image.open('images/fb.jpg')
image.resize((150,150))

In [ ]:
imagetotext('images/fb.jpg', max_new_tokens=300)

In [ ]:
imagetotext(image, max_new_tokens=30)

# 10. 이미지분류

In [ ]:
image = Image.open('images/cat.jpg')
image.resize((150,150))

In [ ]:
imgclassifier = pipeline(task="image-classification")

In [ ]:
imgclassifier(image)